# Datathon 2026 – Task 1: Traffic Speed Forecasting (LightGBM)
**Approach**: Global LightGBM model with spatial neighbour features
- Own speed lags (15 steps) + Neighbour avg lags (5 steps) + Text incident features
- Road ID as categorical feature → model learns per-road patterns automatically
- 3 separate LGBMRegressor models for h5, h10, h15


In [ ]:
import numpy as np
import pandas as pd
import json, os, time
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_DIR = Path('/kaggle/input')
candidates = [d for d in INPUT_DIR.iterdir() if d.is_dir()] if INPUT_DIR.exists() else []
DATA = candidates[0] if candidates else Path('.')

# Robust: handle extra subfolder level (works regardless of how dataset is zipped)
if DATA.exists() and not (DATA / 'train').exists():
    subdirs = [d for d in DATA.iterdir() if d.is_dir()]
    if subdirs:
        DATA = subdirs[0]

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
t0 = time.time()
print(f'Dataset path : {DATA}')
print(f'train/ found : {(DATA / "train").exists()}')
print(f'test/  found : {(DATA / "test").exists()}')
print(f'LightGBM ver : {lgb.__version__}')


In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────────
print('Loading...')
speed_m1  = np.load(DATA / 'train' / 'train_speed_m1_1_11160.npy')         # (11160, 1260)
speed_m2  = np.load(DATA / 'train' / 'train_speed_m2_1_5039.npy')          # (5039,  1260)
test_X    = np.load(DATA / 'test'  / 'test_X_hist.npy').astype(np.float32) # (540, 15, 1260)
train     = np.vstack([speed_m1, speed_m2]).astype(np.float32)              # (16199, 1260)
matrix    = np.load(DATA / 'static' / 'matrix.npy').astype(np.float32)     # (1260, 1260)

with open(DATA / 'train' / 'train_text_m1_1_11160.json') as f: text_m1 = json.load(f)
with open(DATA / 'train' / 'train_text_m2_1_5039.json')  as f: text_m2 = json.load(f)
with open(DATA / 'test'  / 'test_texts.json')             as f: test_texts = json.load(f)

T, R = train.shape   # 16199, 1260
H, F = 15, 3         # history window, forecast horizons
H_n  = 5             # neighbour lag steps
N_TEST = test_X.shape[0]
print(f'train:{train.shape}  test_X:{test_X.shape} | {time.time()-t0:.1f}s')


In [ ]:
# ── Spatial: neighbour average speed ──────────────────────────────────────────
deg      = matrix.sum(axis=1, keepdims=True)
adj_norm = (matrix / np.where(deg == 0, 1, deg)).astype(np.float32)  # row-normalised

neigh      = train  @ adj_norm.T    # (T, R)
neigh_test = test_X @ adj_norm.T    # (540, 15, R)
print(f'Neighbour avg computed | {time.time()-t0:.1f}s')


In [ ]:
# ── Text Features ──────────────────────────────────────────────────────────────
KEYWORDS = ['road closure', 'construction', 'accident', 'prohibit', 'announcement']
N_TEXT   = len(KEYWORDS) + 1  # + incident count

def text_feats(text: str) -> np.ndarray:
    t = text.lower()
    return np.array([float(kw in t) for kw in KEYWORDS] + [float(t.count('.'))],
                   dtype=np.float32)

train_tf = np.zeros((T, N_TEXT), dtype=np.float32)
for i in range(speed_m1.shape[0]):
    k = f'm1_{i+1}'
    if k in text_m1: train_tf[i] = text_feats(text_m1[k])
for i in range(speed_m2.shape[0]):
    k = f'm2_{i+1}'
    if k in text_m2: train_tf[speed_m1.shape[0]+i] = text_feats(text_m2[k])

test_tf = np.zeros((N_TEST, N_TEXT), dtype=np.float32)
for i in range(N_TEST):
    k = f'test_{i:05d}'
    if k in test_texts: test_tf[i] = text_feats(test_texts[k])

print(f'Text features: {N_TEXT} dims | {time.time()-t0:.1f}s')


In [ ]:
# ── Build Training Dataset ──────────────────────────────────────────────────────
# Sample every STEP-th window to stay within ~3M rows
STEP  = 5
n_win = T - H - F + 1
t_idx = np.arange(0, n_win, STEP)  # sampled window starts
r_idx = np.arange(R)               # all roads

print(f'Windows: {len(t_idx)} (every {STEP}th of {n_win})  => ~{len(t_idx)*R/1e6:.1f}M rows')

# Feature names
feat_names = (
    [f'own_lag_{i+1}' for i in range(H)] +
    [f'neigh_lag_{i+1}' for i in range(H_n)] +
    [f'text_{kw.replace(" ","_")}' for kw in KEYWORDS] + ['text_n_incidents'] +
    ['road_id']
)  # 15 + 5 + 6 + 1 = 27 features

def build_train_matrix(f_idx):
    """f_idx: forecast step (0=h5, 1=h10, 2=h15)"""
    rows = []
    targets = []
    for t in t_idx:
        # own speed lags: (H, R) -> each road is a row
        own  = train[t:t+H, :].T            # (R, H)
        nb   = neigh[t+H-H_n:t+H, :].T      # (R, H_n) — last H_n steps of history
        txt  = np.tile(train_tf[t+H], (R,1))  # (R, N_TEXT) — text at target step
        rids = r_idx.reshape(-1, 1)           # (R, 1)
        rows.append(np.hstack([own, nb, txt, rids]))
        targets.append(train[t+H+f_idx, :])   # (R,)
    X = np.vstack(rows).astype(np.float32)
    y = np.concatenate(targets).astype(np.float32)
    return X, y

print('Building train matrix for h5...')
X0, y0 = build_train_matrix(0)
print(f'  X:{X0.shape} | {time.time()-t0:.1f}s')


In [ ]:
# ── Train LightGBM (one model per horizon) ─────────────────────────────────────
lgb_params = dict(
    objective       = 'regression_l1',  # MAE loss
    n_estimators    = 500,
    learning_rate   = 0.05,
    num_leaves      = 127,
    min_child_samples = 50,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 5,
    lambda_l2         = 0.1,
    n_jobs            = -1,
    random_state      = 42,
    verbose           = -1,
)

models = {}
cat_feats = [feat_names.index('road_id')]

for fi, (h, f_idx) in enumerate(zip([5, 10, 15], [0, 1, 2])):
    print(f'Training h{h} model...')
    if fi == 0:
        X, y = X0, y0
    else:
        X, y = build_train_matrix(f_idx)

    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set        = [(X_val, y_val)],
        callbacks       = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
        categorical_feature = cat_feats,
    )
    models[h] = model
    val_mae = np.abs(model.predict(X_val) - y_val).mean()
    print(f'  h{h}: val MAE = {val_mae:.4f} km/h | best iter = {model.best_iteration_} | {time.time()-t0:.1f}s')


In [ ]:
# ── Build Test Features & Predict ──────────────────────────────────────────────
print('Building test features and predicting...')

# test_X:    (540, 15, 1260)  own speed history
# neigh_test:(540, 15, 1260)  neighbour avg history
# test_tf:   (540, N_TEXT)    text features

def build_test_matrix(sample_i):
    """Build feature matrix for one test sample: (R, N_FEAT)."""
    own  = test_X[sample_i, :, :].T             # (R, H)
    nb   = neigh_test[sample_i, -H_n:, :].T      # (R, H_n)
    txt  = np.tile(test_tf[sample_i], (R, 1))    # (R, N_TEXT)
    rids = np.arange(R).reshape(-1, 1)           # (R, 1)
    return np.hstack([own, nb, txt, rids]).astype(np.float32)

# Predict all 540 samples × 1260 roads × 3 horizons
ids, spd = [], []
for si in range(N_TEST):
    X_test_i = build_test_matrix(si)            # (1260, 27)
    s = f'test_{si:05d}'
    for fi, h in enumerate([5, 10, 15]):
        preds = np.clip(models[h].predict(X_test_i), 0.0, 200.0)
        for ri in range(R):
            ids.append(f'{s}_h{h}_r{ri}')
            spd.append(float(preds[ri]))

submission = pd.DataFrame({'id': ids, 'speed': spd})
sub_path = OUT / 'submission.csv'
submission.to_csv(sub_path, index=False)
print(f'Submission saved → {sub_path}')
print(f'Rows: {len(submission):,}')
print(submission['speed'].describe().round(3))
print(f'Total time: {time.time()-t0:.1f}s')


## Summary

**Model**: Global LightGBM (MAE objective) — one model per forecast horizon

**Features (27 total per road-sample)**:
- 15 own speed lag features (t-1 to t-15)
- 5 neighbour average speed lags (using road adjacency matrix)
- 6 text/incident features (road closure, construction, accident, prohibit, announcement, incident count)
- 1 road ID (categorical)

**Training**: ~3.2M samples (every 5th sliding window × 1260 roads)

**Inference**: 540 test samples × 1260 roads × 3 horizons = 2,041,200 predictions
